# datalock v1.0.1 — Guia Completo

> Privacy-by-Design para dados tabulares em Python.
> Renomeada de `logus-lgpd`. O alias `import logus as lg` ainda funciona.

**Seções:**
1. Instalação e setup
2. Leitura universal e big data
3. Manipulação expressiva
4. Privacidade e LGPD
5. Formato `.dlk` — criptografia completa
6. Canary Data — rastreamento de vazamentos
7. `dd.mask_text()` com dados sintéticos
8. `dd.scan_directory()` — inventário de PII
9. Contratos e validação
10. Criptografia assimétrica e expiração
11. Banco de dados
12. Pipeline completo + lineage
13. Relatório LGPD + SQL transpiler
14. Métricas de privacidade
15. Audit webhook
16. Benchmarks


## 1. Instalação e Setup

In [ ]:
!pip install datalock "datalock[sql]" -q

import datalock as dd
import polars as pl
import pandas as pd
import numpy as np
import os, time, json, warnings, tempfile, pathlib
warnings.filterwarnings("ignore")

print(f"datalock  {dd.__version__}")
print(f"polars    {pl.__version__}")
print(f"pandas    {pd.__version__}")

# backward compat — ainda funciona
import logus as lg
print(f"logus shim: {lg.__version__}  (alias para datalock)")


In [ ]:
SALT = dd.generate_salt()   # para mascaramento HMAC
KEY  = dd.generate_salt()   # para criptografia AES-256
assert SALT != KEY, "SALT e KEY devem ser valores DIFERENTES"

# Em produção: use variáveis de ambiente
# SALT = os.environ["DATALOCK_SALT"]
# KEY  = os.environ["DATALOCK_KEY"]

# Configura salt padrão global — dd.mask(df) funciona sem salt= explícito
dd.configure(default_salt=SALT)
print(f"SALT: {SALT[:20]}...  KEY: {KEY[:20]}...")
print("dd.configure(default_salt=SALT) — ativo")


## Dataset de exemplo

In [ ]:
np.random.seed(42)
N = 10_000

df = pl.DataFrame({
    "cpf":          [f"{''.join([str(np.random.randint(0,9)) for _ in range(11)])}" for _ in range(N)],
    "nome":         [f"Cliente {i:05d}" for i in range(N)],
    "email":        [f"cliente{i:05d}@{'gmail' if i%3==0 else 'empresa'}.com.br" for i in range(N)],
    "telefone":     [f"({np.random.randint(11,99)}) 9{np.random.randint(1000,9999)}-{np.random.randint(1000,9999)}" for _ in range(N)],
    "data_nasc":    [f"{np.random.randint(1950,2005)}-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}" for _ in range(N)],
    "cep":          [f"{np.random.randint(1000,99999):05d}-{np.random.randint(0,999):03d}" for _ in range(N)],
    "uf":           [["SP","RJ","MG","RS","BA","PR","SC","GO","PE","CE"][i%10] for i in range(N)],
    "tipo_pessoa":  ["PF" if i%2==0 else "PJ" for i in range(N)],
    "renda_mensal": np.abs(np.random.lognormal(8.5, 0.8, N)).round(2).tolist(),
    "inadimplente": (np.random.rand(N) < 0.12).tolist(),
    "score_credito":np.random.randint(0, 1001, N).tolist(),
})

print(f"Dataset: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"Memória: {df.estimated_size('mb'):.1f} MB")
df.head(3)


## 2. Leitura universal e big data

In [ ]:
# Salva para demonstração
df.write_csv("clientes.csv")
df.write_parquet("clientes.parquet", row_group_size=1_000)

# Leitura universal — qualquer formato retorna pl.DataFrame
for ext in ["csv", "parquet"]:
    t = time.perf_counter()
    r = dd.read(f"clientes.{ext}")
    ms = (time.perf_counter()-t)*1000
    print(f"  .{ext:8}  shape={r.shape}  {ms:.0f}ms")


In [ ]:
# Parâmetros big data — sem OOM
print("header_only (schema, ~0ms):")
info = dd.read("clientes.parquet", header_only=True)
print(f"  n_rows={info['n_rows']:,}  n_cols={info['n_columns']}  rgs={info['n_row_groups']}\n")

print("head=1000:")
r = dd.read("clientes.parquet", head=1_000)
print(f"  shape={r.shape}\n")

print("sample=2000 (row groups aleatórios):")
r = dd.read("clientes.parquet", sample=2_000)
print(f"  shape={r.shape}\n")

print("n_chunks=5, chunks=[2,4] (chunks específicos):")
r = dd.read("clientes.parquet", n_chunks=5, chunks=[2, 4])
print(f"  shape={r.shape}  ({r.shape[0]/N:.0%} dos dados)\n")

print("iter_chunks=True (gerador, nunca carrega tudo):")
gen = dd.read("clientes.parquet", n_chunks=5, iter_chunks=True)
chunks = list(gen)
print(f"  {len(chunks)} chunks  total={sum(len(c) for c in chunks):,} linhas\n")

print("columns pruning (column_pruning real no Parquet):")
r = dd.read("clientes.parquet", columns=["cpf","renda_mensal"])
print(f"  colunas={list(r.columns)}\n")

print("sample= em CSV (aviso honesto):")
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    r = dd.read("clientes.csv", sample=500)
    if w: print(f"  ⚠️  {str(w[0].message)[:80]}...")
print(f"  shape={r.shape}  (head, não aleatório — limitação do CSV)")


## 3. Manipulação expressiva

In [ ]:
# WHERE — todas as sintaxes
casos = [
    ("uf='SP'",           dd.where(df, uf="SP")),
    ("uf in [SP,RJ,MG]",  dd.where(df, uf=["SP","RJ","MG"])),
    ("renda between",     dd.where(df, renda_mensal=(5_000, 15_000))),
    ("renda > 10k",       dd.where(df, renda_mensal=(">", 10_000))),
    ("nome contains",     dd.where(df, nome=("contains", "00100"))),
    ("SP AND PF",         dd.where(df, uf="SP", tipo_pessoa="PF")),
    ("expr Polars",       dd.where(df, dd.col("renda_mensal") > dd.col("score_credito"))),
]
for name, r in casos:
    print(f"  {name:<25} {r.shape[0]:>7,} linhas")


In [ ]:
# ADD COLUMN + WHEN + window functions + shift/lead/lag
df2 = dd.add_column(df,
    imposto       = dd.col("renda_mensal") * 0.275,
    faixa_renda   = dd.when(dd.col("renda_mensal") > 20_000, "A")
                      .when(dd.col("renda_mensal") > 10_000, "B")
                      .when(dd.col("renda_mensal") > 5_000,  "C")
                      .otherwise("D"),
    rank_uf       = dd.col("renda_mensal").rank("dense", descending=True).over("uf"),
    media_uf      = dd.col("renda_mensal").mean().over("uf"),
    renda_lag1    = dd.col("renda_mensal").shift(1),   # período anterior
)
novas = [c for c in df2.columns if c not in df.columns]
print("Colunas adicionadas:", novas)
print(df2.select(["uf","renda_mensal","faixa_renda","rank_uf","renda_lag1"]).head(4))


In [ ]:
# SHIFT / LAG / LEAD — séries temporais
df_ts = pl.DataFrame({
    "mes":   ["Jan","Fev","Mar","Abr","Mai","Jun"],
    "renda": [5000., 5200., 4800., 5100., 5300., 5150.]
})
df_ts2 = dd.add_column(df_ts,
    renda_ant  = dd.col("renda").shift(1),
    renda_prox = dd.col("renda").shift(-1),
    var_mes    = dd.col("renda") - dd.col("renda").shift(1),
)
print("shift / lag / lead:")
print(df_ts2)

# EXPLODE — listas em linhas
df_tags = pl.DataFrame({
    "id":   [1, 2, 3],
    "tags": [["lgpd","privacidade"], ["python","polars","datalock"], ["dados"]],
})
exploded = dd.explode(df_tags, "tags")
print(f"\nexplode: {df_tags.shape} → {exploded.shape} (6 linhas)")
print(exploded)


In [ ]:
# GROUPBY completo
resultado = dd.groupby(df, "uf", {
    "n":           ("*",             "count"),
    "media_renda": ("renda_mensal",  "mean"),
    "inadimplentes":("inadimplente", "sum"),
    "score_medio": ("score_credito", "mean"),
}, having={"n": (">", 800)}, sort="media_renda", desc=True, limit=5)
print(resultado)


In [ ]:
# PIPELINE FLUENTE
result = (
    dd.pipe("clientes.parquet")
    .where(uf=["SP","RJ"], tipo_pessoa="PF")
    .add_column(
        imposto  = dd.col("renda_mensal") * 0.275,
        faixa    = dd.when(dd.col("renda_mensal") > 10_000, "alta")
                     .when(dd.col("renda_mensal") > 5_000, "media")
                     .otherwise("baixa"),
    )
    .mask(salt=SALT)
    .groupby("faixa", {"n": ("*","count"), "media": ("renda_mensal","mean")})
    .sort("media", desc=True)
    .collect()
)
print(result)


## 4. Privacidade e LGPD

In [ ]:
# SCAN — detecção automática de PII
reports = dd.scan(df)

print(f"Colunas PII detectadas: {len(reports)}\n")
print(f"{'Coluna':<20} {'Tipo':<22} {'Risco':<8} {'Match':>6}  {'Estratégia'}")
print("─"*72)
for col, r in reports.items():
    flag = {"high":"🔴","medium":"🟡","low":"🟢"}.get(r.risk_level.value,"•")
    print(f"{col:<20} {r.pii_type.value:<22} {flag} {r.risk_level.value:<6} "
          f"{r.match_ratio:>5.0%}  {r.mask_strategy.value}")


In [ ]:
# CUSTOM PATTERNS — identificadores proprietários
from logus.detectors.fast_scan import FastPIIScanner   # ou from datalock...

df_corp = pl.DataFrame({
    "protocolo":  [f"CTR-{i:08d}" for i in range(100)],
    "descricao":  [f"Contrato {i}" for i in range(100)],
    "renda":      [float(i*1000) for i in range(100)],
})

reports_custom = dd.scan(df_corp, custom_patterns={
    "num_contrato": r"^CTR-[0-9]{8}$",
})
print("Detecção com padrões customizados:")
for col, r in reports_custom.items():
    print(f"  {col}: {r.pii_type.value} ({r.match_ratio:.0%} match)")


In [ ]:
# MASK — mascaramento HMAC determinístico
t0 = time.perf_counter()
df_safe = dd.mask(df, salt=SALT)
t_mask  = (time.perf_counter()-t0)*1000
print(f"mask({N:,} linhas): {t_mask:.0f}ms")
print(f"Tipo retornado: {type(df_safe).__name__}")

# Determinismo — JOIN-safe
cpf_original = df["cpf"][0]
token1 = df_safe["cpf"][0]
token2 = dd.mask(df.head(1), salt=SALT)["cpf"][0]
print(f"\nDeterminismo: {token1 == token2}  ← mesmo CPF, mesmo SALT → mesmo token")
print(f"  Original: {cpf_original}")
print(f"  Token:    {token1}")

# LazyFrame — permanece lazy
lf_safe = dd.mask(df.lazy(), salt=SALT)
print(f"\nmask(LazyFrame) → {type(lf_safe).__name__}  (lazy até .collect())")


## 5. Formato `.dlk` — criptografia AES-256-GCM

In [ ]:
# Sem criptografia (dados já anonimizados)
dd.store(df_safe, "dados_dev.dlk", overwrite=True)

# Com criptografia
dd.store(df, "dados_raw.dlk", key=KEY, overwrite=True)

# Criptografia + mascaramento
dd.store(df, "dados_masked.dlk", key=KEY, salt=SALT, overwrite=True)

# Verifica MAGIC bytes
with open("dados_raw.dlk","rb") as f:
    magic = f.read(5)
print(f"MAGIC bytes: {magic}  ← b'DLOCK'")

for fname in ["dados_dev.dlk","dados_raw.dlk","dados_masked.dlk"]:
    r = dd.read(fname) if "dev" in fname else dd.read(fname, key=KEY)
    print(f"  {fname:30} shape={r.shape}")


In [ ]:
# METADATA INDEX — sem decifrar o payload
info = dd.inspect("dados_masked.dlk", key=KEY)
print("Metadados sem decifrar payload:")
for k, v in info.items():
    if k not in ("column_stats","metadata","schema"):
        print(f"  {k}: {v}")


In [ ]:
# MULTI-FRAME — múltiplos DataFrames num único arquivo
dd.store(
    {"clientes": df.to_pandas(), "pedidos": df.select(["cpf","renda_mensal"]).head(1000).to_pandas()},
    "base_completa.dlk",
    key=KEY, overwrite=True,
)
frames = dd.read("base_completa.dlk", key=KEY)
for name, frame in frames.items():
    print(f"  Frame '{name}': {frame.shape}")


In [ ]:
# Comparação de tamanhos
import os
df.write_csv("sz.csv"); df.write_parquet("sz.parquet")
dd.store(df.to_pandas(), "sz.dlk", key=KEY, overwrite=True)

print(f"{'Formato':<28} {'Tamanho':>12}")
print("─"*42)
for label, path in [(".csv",  "sz.csv"), (".parquet (zstd)", "sz.parquet"), (".dlk (AES+zstd)", "sz.dlk")]:
    sz = os.path.getsize(path)
    print(f"{label:<28} {sz/1024:>10.1f} KB")


## 6. Canary Data — rastreamento transparente de vazamentos

In [ ]:
# O usuário NUNCA vê as linhas canary — injetadas e removidas automaticamente

# Salva com canary
dd.store(df.to_pandas(), "clientes_canary.dlk", key=KEY,
         canary=True, pipeline_id="pipeline_crm_jan2025",
         overwrite=True)

# Lê — canary removido silenciosamente
df_back = dd.read("clientes_canary.dlk", key=KEY)

print(f"Shape original:  {df.shape}")
print(f"Shape ao ler:    {df_back.shape}  ← idêntico (canary stripped)")
print(f"Sem col canary:  {'__canary_sig__' not in df_back.columns}")


In [ ]:
# Inspeciona metadados do canary (sem decifrar payload)
info = dd.inspect("clientes_canary.dlk", key=KEY)
ci   = dd.canary_info(info)

print("Canary metadata no header:")
print(f"  pipeline_id:  {ci['pipeline_id']}")
print(f"  fingerprints: {ci['fingerprints']}")
print(f"  injected_at:  {ci['injected_at'][:19]}")
print(f"  n_rows:       {ci['n_canary_rows']}")


In [ ]:
# Simula descoberta de um token canary num dump de breach
# "encontrei canary.1ba472d8@datalock.internal num dump da dark web"
fp = ci["fingerprints"][0]
email_canary = f"canary.{fp[:8]}@datalock.internal"
print(f"Token canary: {email_canary}\n")

# Lookup: qual pipeline gerou esse token?
result = dd.canary_check(email_canary)
if result:
    print("🚨 Canary detectado! Rastreamento:")
    print(f"   pipeline_id:  {result['pipeline_id']}")
    print(f"   arquivo:      {result['filepath']}")
    print(f"   injetado em:  {result['injected_at'][:19]}")
else:
    # Pode não estar no manifesto local deste ambiente
    result_fp = dd.canary_check(fp)
    print(f"Lookup por fingerprint: {result_fp}")


## 7. `dd.mask_text()` — mascaramento de texto livre

In [ ]:
texto = (
    "Cliente CPF 111.444.777-35 solicitou crédito. "
    "Email: joao.silva@empresa.com.br  Tel: (11) 98765-4321  "
    "CEP: 01310-100  Nascimento: 15/03/1985"
)

print(f"Original:\n  {texto}\n")

for strategy in ("redact", "hash", "partial", "semantic"):
    resultado = dd.mask_text(texto, salt=SALT, strategy=strategy)
    print(f"{strategy:10}: {resultado}")


In [ ]:
# Strategy 'semantic' — dados sintéticos válidos, sem faker obrigatório

# SyntheticGenerator — pode usar diretamente
gen = dd.SyntheticGenerator(seed=42)
print("Dados sintéticos gerados internamente (sem faker):")
print(f"  CPF:            {gen.cpf()}   ← dígitos verificadores corretos")
print(f"  CNPJ:           {gen.cnpj()}")
print(f"  Email:          {gen.email()}")
print(f"  Nome:           {gen.nome()}")
print(f"  Telefone:       {gen.telefone()}")
print(f"  CEP:            {gen.cep()}")
print(f"  Data nascimento:{gen.data_nascimento()}")

# Determinismo — mesmo seed → mesmo output (preserva JOINs)
g1 = dd.SyntheticGenerator(seed=42); g2 = dd.SyntheticGenerator(seed=42)
print(f"\nDeterminismo: {g1.cpf() == g2.cpf()}  ← mesmo seed → mesmo CPF falso")

# Com Faker instalado: pip install datalock[synthetic]
# SyntheticGenerator usa Faker pt_BR automaticamente
print("\n(pip install datalock[synthetic] → usa Faker pt_BR para maior qualidade)")


## 8. `dd.scan_directory()` — inventário de PII

In [ ]:
# Cria estrutura de exemplo
os.makedirs("dados_empresa/clientes", exist_ok=True)
os.makedirs("dados_empresa/financeiro", exist_ok=True)
os.makedirs("dados_empresa/config", exist_ok=True)

df.write_csv("dados_empresa/clientes/clientes_2025.csv")
df.select(["cpf","renda_mensal","inadimplente"]).write_parquet(
    "dados_empresa/financeiro/analise.parquet")
pd.DataFrame({"config":["a","b"],"valor":[1,2]}).to_csv(
    "dados_empresa/config/settings.csv", index=False)

print("Estrutura criada:")
for p in pathlib.Path("dados_empresa").rglob("*"):
    if p.is_file():
        print(f"  {p}")


In [ ]:
# Inventário completo — recursivo
inventory = dd.scan_directory("dados_empresa/",
                               recursive=True, verbose=False)

print(inventory.summary())


In [ ]:
# Filtro por risco
inv_high = dd.scan_directory("dados_empresa/", min_risk="high")
print(f"Arquivos de alto risco: {len(inv_high.files_with_pii)}")
for path, fi in inv_high.items():
    print(f"  {pathlib.Path(path).name:<35} {fi.max_risk:<8} → {list(fi.pii_columns.keys())}")


In [ ]:
# Relatório HTML e JSON
html = inventory.to_html("inventario_pii.html")
js   = inventory.to_json("inventario_pii.json")
print(f"HTML: {len(html)} chars  → inventario_pii.html")
print(f"JSON: {len(js)} chars  → inventario_pii.json")

# Acesso programático
for path, fi in inventory.items():
    if fi.has_pii:
        print(f"\n  {pathlib.Path(path).name}: {fi.n_pii_columns} colunas PII  max_risk={fi.max_risk}")
        for col, info in fi.pii_columns.items():
            print(f"    {col:<20} {info['pii_type']:<20} {info['risk']}")


## 9. Contratos e validação

In [ ]:
LISTA_UFS = ["SP","RJ","MG","RS","BA","PR","SC","GO","PE","CE"]

contrato = dd.contract({
    "cpf":          {"type":"str",   "not_null":True, "pii":"CPF",   "mask":"hash"},
    "email":        {"type":"str",   "contains":"@",  "pii":"email", "mask":"hash"},
    "renda_mensal": {"type":"float", "min":0, "max":500_000,
                     "pii":"numerico", "mask":"mock_numeric"},
    "uf":           {"type":"str",   "in":LISTA_UFS},
    "score_credito":{"type":"int",   "min":0, "max":1000},
}, name="clientes_v2", version="2.0")

print(repr(contrato))
result = contrato.apply(df.to_pandas(), salt=SALT, strict=False)
print(f"passed={result.passed}  pii={result.pii_columns}  t={result.elapsed_ms:.0f}ms")


In [ ]:
# save / load / diff / JSON Schema
contrato.save("/tmp/clientes_v2.contract.json")
contrato_back = dd.DataContract.load("/tmp/clientes_v2.contract.json")

contrato_v3 = dd.contract({
    "cpf":          {"type":"str","pii":"CPF","mask":"hash"},
    "email":        {"type":"str","contains":"@","pii":"email","mask":"hash"},
    "renda_mensal": {"type":"float","min":0,"max":800_000,"pii":"numerico","mask":"mock_numeric"},
    "uf":           {"type":"str","in":LISTA_UFS},
    "nova_coluna":  {"type":"str"},     # adicionada
    # score_credito removido ← BREAKING
}, name="clientes_v3", version="3.0")

diff = contrato.diff(contrato_v3)
print(f"Breaking changes: {diff.has_breaking_changes}")
print(diff.report())


In [ ]:
# validate_schema + save/load rules
r = dd.validate_schema(df.to_pandas(),
    required_columns  = ["cpf","email","renda_mensal","uf"],
    forbidden_columns = ["senha","token_acesso"],
    min_rows          = 1,
)
print(f"validate_schema: passed={r.passed}  rules={r.n_rules}")

regras = {"cpf":{"not_null":True},"renda_mensal":{"min":0,"max":500_000}}
dd.save_rules(regras, "/tmp/regras_v2.json")
loaded = dd.load_rules("/tmp/regras_v2.json")
print(f"save/load rules: match={loaded==regras}")


## 10. Criptografia assimétrica e expiração de arquivo

In [ ]:
# Gera par de chaves
priv, pub = dd.generate_keypair("ec")   # EC P-256
print(f"EC keypair: {type(priv).__name__} / {type(pub).__name__}")

# Salva e carrega
dd.save_keypair(priv, "/tmp/minha_chave.pem", pub, "/tmp/minha_pub.pem")
priv_l = dd.load_private_key("/tmp/minha_chave.pem")
pub_l  = dd.load_public_key("/tmp/minha_pub.pem")
print("Save/load keypair: OK")

# Store com chave pública — destinatário decifra com privada
dd.store(df.head(100).to_pandas(), "assimetrico.dlk", public_key=pub, overwrite=True)
df_back = dd.read("assimetrico.dlk", private_key=priv)
print(f"Assimétrico store/read: {df_back.shape}")


In [ ]:
# Expiração de arquivo — LGPD Art. 16
dd.store(df.head(10).to_pandas(), "expira_2099.dlk", key=KEY,
         expires_at="2099-12-31", overwrite=True)
df_ok = dd.read("expira_2099.dlk", key=KEY)
print(f"Válido (2099): {df_ok.shape}")

dd.store(df.head(10).to_pandas(), "expirado.dlk", key=KEY,
         expires_at="2020-01-01", overwrite=True)
try:
    dd.read("expirado.dlk", key=KEY)
    print("FALHOU: deveria ter levantado ExpiredFileError")
except dd.ExpiredFileError as e:
    print(f"ExpiredFileError: {str(e)[:70]}")


## 11. Banco de dados — `dd.db()`

In [ ]:
import sqlite3

db_path = "/tmp/clientes_demo.db"
conn = sqlite3.connect(db_path)
conn.execute("DROP TABLE IF EXISTS clientes")
conn.execute("CREATE TABLE clientes (cpf TEXT, nome TEXT, renda REAL, uf TEXT)")
for i in range(200):
    conn.execute("INSERT INTO clientes VALUES (?,?,?,?)",
                 (f"{i:011d}", f"Cliente {i}", float(i*100), "SP" if i%2==0 else "RJ"))
conn.commit(); conn.close()

banco = dd.db(f"sqlite:///{db_path}", salt=SALT)
print(f"Conexão: {banco}")
print(f"Tabelas: {banco.tables()}")
print(f"Schema:  {banco.schema('clientes')}")


In [ ]:
# Leitura com mascaramento automático
df_db = dd.read(banco, "clientes")
print(f"dd.read(banco, 'clientes'): {df_db.shape}  tipo={type(df_db).__name__}")

# SQL direto
df_sql = dd.read(banco, "SELECT * FROM clientes WHERE uf='SP' LIMIT 10")
print(f"SQL direto: {df_sql.shape}")

# Amostra SEM mascaramento (para inspeção)
sample = banco.sample_table("clientes", n=3)
print(f"sample_table (sem mask): {sample['cpf'].to_list()}")


In [ ]:
# create_table + upsert
df_new = pl.DataFrame({
    "cpf":  ["00000000001","00000000002","00000000999"],
    "nome": ["Ana","Bruno","Carlos"],
    "renda":[9000.,12000.,5000.],
    "uf":   ["SP","RJ","MG"],
})
banco.create_table(df_new.to_pandas(), "clientes_v2", if_exists="replace")

df_upd = pl.DataFrame({
    "cpf":  ["00000000001","00000000999","00000001000"],
    "nome": ["Ana Silva","Carlos Novo","Daniela"],
    "renda":[10000.,5500.,8000.],
    "uf":   ["SP","MG","RS"],
})
banco.upsert(df_upd.to_pandas(), "clientes_v2", on="cpf")
final = dd.read(banco, "SELECT * FROM clientes_v2")
print(f"Após upsert: {final.shape}  nomes={final['nome'].to_list()}")


## 12. `dd.process()` — pipeline completo

In [ ]:
t0 = time.perf_counter()
result = dd.process(
    "clientes.parquet",
    salt       = SALT,
    key        = KEY,
    output     = "clientes_safe.dlk",
    overwrite  = True,
    where      = {"uf": ["SP","RJ","MG"]},
    rules      = {
        "cpf":          {"not_null": True},
        "email":        {"contains": "@"},
        "renda_mensal": {"min": 0, "max": 500_000},
    },
)
t_total = (time.perf_counter()-t0)*1000

result.print_summary()
print(f"\nDetalhes:")
print(f"  df tipo:       {type(result.df).__name__}  {result.df.shape}")
print(f"  privacy_score: {result.privacy_score}/100")
print(f"  validation:    passed={result.validation.passed}")
print(f"  lineage ops:   {result.lineage.n_operations}")
print(f"  output:        {result.output_path}")
print(f"  tempo total:   {t_total:.0f}ms")


## 13. Relatório LGPD + SQL Transpiler

In [ ]:
reports = dd.scan(df)
report  = dd.compliance_report(
    df, reports,
    dataset_name = "Clientes Q1 2025",
    organization = "Empresa Demo Ltda.",
    extra_notes  = "Revisado pelo DPO.",
)
html = report.to_html("lgpd_relatorio.html")
print(report.to_text()[:600])
print(f"\nHTML: {len(html)} chars → lgpd_relatorio.html")


In [ ]:
# SQL Transpiler — mascara dados direto no banco
safe_sql = dd.mask_sql(
    "SELECT cpf, nome, email, renda_mensal, uf FROM clientes WHERE uf = 'SP'",
    reports  = reports,
    dialect  = "postgresql",
    salt     = "meu-salt-do-banco",
    annotate = True,
)
print(safe_sql[:500])


## 14. Métricas de privacidade

In [ ]:
# k-anonimato
df_anon = dd.add_column(df,
    faixa_score = dd.when(dd.col("score_credito") > 700, "alto")
                    .when(dd.col("score_credito") > 400, "medio")
                    .otherwise("baixo")
)
kanon = dd.check.kanon(
    df_anon,
    quasi_identifiers=["uf","tipo_pessoa","faixa_score"],
    target_k=5,
)
print(f"k-anonimato: k={kanon.k_anonymity.k_value}  ANPD(k≥5)={kanon.compliant_anpd}")

risk    = dd.check.risk(df_safe, quasi_identifiers=["uf","tipo_pessoa"])
utility = dd.check.utility(df.to_pandas(), df_safe.to_pandas())
profile = dd.profile(df)
ps      = profile["privacy_score"]

print(f"Risk score:    {risk.risk_score:.2%}  nível={risk.risk_level}")
print(f"Utilidade:     {utility.overall_score:.0%}")
print(f"Privacy score: {ps['total']}/100 [{ps['grade']}]  — {ps['recommendation']}")


## 15. Audit Webhook

In [ ]:
# Configura webhook — POST não-bloqueante por operação
# Em produção: endpoint Slack, SIEM, Datadog, etc.
dd.configure(
    audit_webhook = "https://hooks.slack.com/services/T.../B.../xxx",  # exemplo
    # audit_path  = "./audit/",  # trilha local também
)

# Cada mask(), scan(), store() dispara um POST em background thread
# Se o endpoint falhar: pipeline continua sem erro
df_safe2 = dd.mask(df.head(100), salt=SALT)  # silently fires webhook

print("Webhook configurado. POST assíncrono por operação.")
print("Payload enviado:")
import json
payload_exemplo = {
    "source": "datalock-audit",
    "version": dd.__version__,
    "event": {
        "column":        "cpf",
        "technique":     "hmac_sha256",
        "rows_affected": 100,
        "timestamp":     "2025-01-15T10:30:00Z",
        "status":        "success",
    }
}
print(json.dumps(payload_exemplo, indent=2))


## 16. Benchmarks de performance

In [ ]:
from datalock.detectors.fast_scan import FastPIIScanner
from datalock.detectors.pii_detector import PIIDetector

N_b = 50_000
df_b = pl.DataFrame({
    "cpf":   [f"{i:011d}" for i in range(N_b)],
    "email": [f"u{i}@x.com" for i in range(N_b)],
    "renda": np.random.lognormal(8.5,.8,N_b).tolist(),
    "uf":    (["SP","RJ","MG","RS"]*(N_b//4+1))[:N_b],
    "nome":  [f"C {i}" for i in range(N_b)],
    "cep":   [f"{i:05d}-{i%999:03d}" for i in range(N_b)],
})

# Warmup
FastPIIScanner().detect_dict(df_b)
dd.mask(df_b.head(1000), salt=SALT)

results = {}

# scan
t=time.perf_counter(); FastPIIScanner().detect_dict(df_b); tf=(time.perf_counter()-t)*1000
t=time.perf_counter(); PIIDetector().detect(df_b.to_pandas()); ts=(time.perf_counter()-t)*1000
results["scan (FastPIIScanner)"]  = f"{tf:.0f}ms"
results["scan (PIIDetector)"]     = f"{ts:.0f}ms  ({ts/max(tf,.1):.1f}× slower)"

# mask
t=time.perf_counter(); dd.mask(df_b, salt=SALT); results["mask 50k"]=(f"{(time.perf_counter()-t)*1000:.0f}ms")
t=time.perf_counter(); dd.mask(df_b.lazy(),salt=SALT).collect(); results["mask 50k (LazyFrame)"]=(f"{(time.perf_counter()-t)*1000:.0f}ms")

# process
t=time.perf_counter(); dd.process(df_b,salt=SALT); results["process 50k"]=(f"{(time.perf_counter()-t)*1000:.0f}ms")

# synthetic
t=time.perf_counter()
g=dd.SyntheticGenerator(seed=42)
for _ in range(1000): g.cpf()
results["SyntheticGenerator 1k CPFs"]=(f"{(time.perf_counter()-t)*1000:.1f}ms")

# mask_text
import random; texts=[f"CPF {random.randint(10000000000,99999999999)}, email u{i}@co.com" for i in range(100)]
t=time.perf_counter()
for txt in texts: dd.mask_text(txt,salt=SALT,strategy='semantic')
results["mask_text semantic 100×"]=(f"{(time.perf_counter()-t)*1000:.0f}ms")

print(f"Benchmarks ({N_b:,} linhas):")
for name,val in results.items():
    print(f"  {name:<35} {val}")


## Limpeza

In [ ]:
import glob, shutil
for f in glob.glob("*.csv")+glob.glob("*.parquet")+glob.glob("*.dlk")+glob.glob("*.json")+glob.glob("*.html")+glob.glob("*.pem"):
    try: os.remove(f)
    except: pass
try: shutil.rmtree("dados_empresa")
except: pass
print("Arquivos temporários removidos.")
print("\n✅ Notebook completo — todas as funcionalidades da datalock v1.0.1 demonstradas.")


---

## Referência rápida — datalock v1.0.1

```python
import datalock as dd
import os

SALT = os.environ["DATALOCK_SALT"]
KEY  = os.environ["DATALOCK_KEY"]

# I/O
df = dd.read("arquivo.qualquer")
dd.store(df_safe, "out.dlk", key=KEY, canary=True, expires_at="2025-12-31")

# PII
reports = dd.scan(df, custom_patterns={"contrato": r"^CTR-[0-9]{8}$"})
df_safe = dd.mask(df, salt=SALT)         # pd→pd / pl→pl / lazy→lazy

# Texto livre
dd.mask_text(text, salt=SALT, strategy="semantic")  # CPF falso válido

# Diretório
dd.scan_directory("./dados/").to_html("report.html")

# Canary
dd.store(df, "f.dlk", key=KEY, canary=True)          # inject silently
df = dd.read("f.dlk", key=KEY)                        # strip silently
dd.canary_check("canary.1ba472d8@datalock.internal")  # lookup

# Pipeline
result = dd.process("in.csv", salt=SALT, key=KEY, output="out.dlk")

# Banco
banco = dd.db("postgresql://...", salt=SALT)
df = dd.read(banco, "tabela")
banco.create_table(df, "nova"); banco.upsert(df_new, "nova", on="cpf")

# Webhook
dd.configure(audit_webhook="https://hooks.slack.com/...")

# Contrato
c = dd.contract({"cpf":{"pii":"CPF","mask":"hash"}})
c.apply(df, salt=SALT).raise_if_failed()
```
